# M1 — FROZEN BioViL-T feature extraction (phase_1)

Extracts one `<image_id>.pt` (`[197, 512]` float16) per CXR — the **exact cache contract** that
`phase_3/src/features.py` loads. row 0 = projected global embedding; rows 1..196 = projected
`[512,14,14]` patch grid flattened `index = y*14+x`.

Run order: **config -> clone -> rclone OAuth -> reference sanity (go/no-go) -> extract (resumable) -> verify**.
Internet ON. GPU ON (T4/P100). The reference cell reproduces a known `docs/*.pt` and asserts
`cosine ~ 1` BEFORE the long run — if it fails, STOP and fix the transform/encoder, don't extract 250k.


## 1. Config


In [ ]:
import os
# ===== code from GitHub (cloned next cell) =====
REPO = "/kaggle/working/repo"
PHASE1 = f"{REPO}/phase_1"

# ===== durable cache on Google Drive via YOUR OAuth token (see rclone cell) =====
DRIVE_FOLDER_ID = "1c6AJ3fjsL449kiMK4xiXfnzwzA4gIo0O"   # CHEX-DATA folder id (same as phase_2)
RCLONE_REMOTE   = "dhint:biovilt_features"               # = CHEX-DATA/biovilt_features

# ===== extraction =====
BATCH      = 48      # images per forward (T4 16GB is comfortable)
FLUSH_EVERY= 1000    # push this many .pt to Drive then free local /kaggle/working
LIMIT      = 0       # >0 = stop after N new images (smoke test); 0 = all

for k, v in dict(REPO=REPO, PHASE1=PHASE1, DRIVE_FOLDER_ID=DRIVE_FOLDER_ID,
                 RCLONE_REMOTE=RCLONE_REMOTE, BATCH=BATCH, FLUSH_EVERY=FLUSH_EVERY,
                 LIMIT=LIMIT).items():
    os.environ[k] = str(v)
print("remote =", RCLONE_REMOTE, "| BATCH =", BATCH, "| LIMIT =", LIMIT)

In [ ]:
# --- get the code from GitHub. Internet ON. ---
!rm -rf /kaggle/working/repo && git clone -q https://github.com/hiennguyendang/phase_2_3_4_5 /kaggle/working/repo
!ls /kaggle/working/repo/phase_1/scripts

## 2. Resolve input mounts (auto-detect)
`data/` is gitignored, so the manifest / pairs / labels / images come from **Kaggle datasets**,
not the repo clone. Add them to the notebook: `nguynnghin/mimic-cxr-448`, and the m3/m4 label
datasets that carry `manifest.jsonl`, `m3_pairs.jsonl`, `boxes.npy`.


In [ ]:
import glob, os
from pathlib import Path

# Your datasets live under datasets/nguynnghin/<slug>; also fall back to the classic
# /kaggle/input/<slug> mount. Targeted + lazy (iglob) so we DON'T recursively walk the 377k jpgs.
BASES = ['/kaggle/input/datasets/nguynnghin', '/kaggle/input']

def resolve_file(slug, name):
    for b in BASES:                                   # direct, then one Kaggle wrap level
        for pat in (f'{b}/{slug}/{name}', f'{b}/{slug}/*/{name}'):
            hit = next(iter(glob.iglob(pat)), None)
            if hit:
                return hit
    for b in BASES:                                   # last resort: search only inside the slug
        hit = next(iter(glob.iglob(f'{b}/{slug}/**/{name}', recursive=True)), None)
        if hit:
            return hit
    return None

def resolve_images(slug):
    for b in BASES:                                   # the dir directly holding p10/p10000032/*.jpg
        for cand in [f'{b}/{slug}'] + sorted(glob.glob(f'{b}/{slug}/*')):
            if os.path.isdir(cand) and next(iter(glob.iglob(f'{cand}/p*/p*/*.jpg')), None):
                return cand
    return None

IMAGES_ROOT = resolve_images('mimic-cxr-448')
MANIFEST    = resolve_file('m3-labels', 'manifest.jsonl')
BOXES       = resolve_file('m3-labels', 'boxes.npy')
PAIRS       = resolve_file('m4-labels', 'm3_pairs.jsonl')
LABELS_DIR  = str(Path(BOXES).parent) if BOXES else ''
for k, v in dict(IMAGES_ROOT=IMAGES_ROOT, MANIFEST=MANIFEST, PAIRS=PAIRS,
                 BOXES=BOXES, LABELS_DIR=LABELS_DIR).items():
    os.environ[k] = str(v) if v else ''
    print(f'{k:12} = {v}')
assert IMAGES_ROOT and MANIFEST and PAIRS and BOXES, \
    'not found — check the dataset slugs (mimic-cxr-448 / m3-labels / m4-labels) are attached'

## 3. rclone + Drive auth (YOUR OAuth token)
Install rclone, then configure remote `dhint` from the Kaggle Secret `GDRIVE_TOKEN`
(`rclone authorize "drive"`). **Not** a service account — an SA has no My-Drive quota and 403s
on upload. Optional `GDRIVE_CLIENT_ID/SECRET` give a private query quota for the many small files.


In [ ]:
%%bash
set -e
if ! command -v rclone >/dev/null 2>&1; then
  cd /kaggle/working && curl -sLO https://downloads.rclone.org/rclone-current-linux-amd64.zip
  unzip -q -o rclone-current-linux-amd64.zip && cp rclone-*-linux-amd64/rclone /usr/local/bin/ && chmod +x /usr/local/bin/rclone
fi
rclone version | head -1

In [ ]:
import os
# Drive remote 'dhint' via YOUR OAuth token (NOT a service account: SA has no storage quota and
# 403s on upload to My Drive). Token = `rclone authorize "drive"` -> Kaggle Secret GDRIVE_TOKEN.
# Graceful: if missing/unwritable, extraction still runs but WITHOUT Drive sync/resume.
SYNC_OK = "0"
try:
    from kaggle_secrets import UserSecretsClient
    sec = UserSecretsClient()
    token = sec.get_secret("GDRIVE_TOKEN").strip()
    os.environ["RCLONE_CONFIG_DHINT_TYPE"] = "drive"
    os.environ["RCLONE_CONFIG_DHINT_TOKEN"] = token
    os.environ["RCLONE_CONFIG_DHINT_SCOPE"] = "drive"
    os.environ["RCLONE_CONFIG_DHINT_ROOT_FOLDER_ID"] = os.environ["DRIVE_FOLDER_ID"]
    os.environ.pop("RCLONE_CONFIG_DHINT_SERVICE_ACCOUNT_FILE", None)  # drop stale SA
    for k_sec, k_env in [('GDRIVE_CLIENT_ID', 'RCLONE_CONFIG_DHINT_CLIENT_ID'),
                         ('GDRIVE_CLIENT_SECRET', 'RCLONE_CONFIG_DHINT_CLIENT_SECRET')]:
        try:
            os.environ[k_env] = sec.get_secret(k_sec).strip()
        except Exception:
            pass
    using_own = 'own client' if os.environ.get('RCLONE_CONFIG_DHINT_CLIENT_ID') else 'rclone shared client'
    remote = os.environ["RCLONE_REMOTE"]
    if os.system('rclone mkdir "%s"' % remote) == 0 and \
       os.system('echo ok | rclone rcat "%s/_write_test.txt"' % remote) == 0:
        SYNC_OK = "1"
        os.system('rclone delete "%s/_write_test.txt" 2>/dev/null' % remote)
        print(f"remote OK (OAuth, write verified, {using_own}) ->", remote)
    else:
        print("[WARN] Drive write FAILED -> check GDRIVE_TOKEN (write scope) + DRIVE_FOLDER_ID")
except Exception as e:
    print("[WARN] GDRIVE_TOKEN secret not set -> NO Drive sync:", e)
os.environ["SYNC_OK"] = SYNC_OK
print("SYNC_OK =", SYNC_OK)

## 4. Install BioViL-T (`hi-ml-multimodal`)


In [ ]:
!pip install -q hi-ml-multimodal 2>/dev/null || pip install -q git+https://github.com/microsoft/hi-ml.git#subdirectory=hi-ml-multimodal
import health_multimodal, torch
print("health_multimodal", health_multimodal.__version__ if hasattr(health_multimodal, "__version__") else "ok",
      "| torch", torch.__version__, "| cuda", torch.cuda.is_available())

## 5. Build the worklist (manifest U prior_image_id)


In [ ]:
!python $PHASE1/scripts/1-build_worklist.py \
    --images-root $IMAGES_ROOT --manifest $MANIFEST --pairs $PAIRS \
    --out /kaggle/working/worklist.jsonl

## 6. Reference sanity — GO / NO-GO (run this FIRST)
Reproduce the bundled `docs/<id>.pt` with the live encoder and assert `cosine ~ 1`. This proves the
**model variant + preprocessing + flatten order** all match the existing collaborator cache. If it
FAILS, flip `TRANSFORM_MODE` in `phase_1/src/config.py` to `resize_crop` (or check the encoder) and
re-run — do **not** extract 250k images until this passes.


In [ ]:
import glob, os
os.environ['REF'] = glob.glob(f'{PHASE1}/reference/MIMIC_*.pt')[0]
print('reference =', os.environ['REF'])
!python $PHASE1/scripts/3-verify_features.py \
    --reference $REF --images-root $IMAGES_ROOT \
    --features-root /kaggle/working/features --manifest $MANIFEST --pairs $PAIRS \
    --labels-dir $LABELS_DIR --region 'right lung' --align-png /kaggle/working/alignment_check.png

## 7. Extract (resumable)
Writes `<id>.pt`, flushes to Drive every `FLUSH_EVERY`, deletes local to free space. Re-run after a
session dies — it skips image_ids already on Drive (`rclone lsf`) or staged locally.


In [ ]:
!python $PHASE1/scripts/2-extract_features.py \
    --worklist /kaggle/working/worklist.jsonl --out-dir /kaggle/working/features \
    --remote $RCLONE_REMOTE --device cuda --batch $BATCH --flush-every $FLUSH_EVERY --limit $LIMIT

## 8. Verify
Structure / naming / coverage + the alignment overlay. (Coverage counts the **local** staging dir;
for full-cache coverage, `rclone copy $RCLONE_REMOTE /kaggle/working/features` a sample back first.)


In [ ]:
!python $PHASE1/scripts/3-verify_features.py \
    --features-root /kaggle/working/features --labels-dir $LABELS_DIR \
    --images-root $IMAGES_ROOT --manifest $MANIFEST --pairs $PAIRS \
    --region 'right lung' --align-png /kaggle/working/alignment_check.png
from IPython.display import Image as IPImage
IPImage('/kaggle/working/alignment_check.png')